# 15a. Phân tích FPR trên val (task 5.3 tiêu chí 2, GĐ F phần 1)

Trên val của NF-ToN-IoT-v3 và NF-BoT-IoT-v3, với LightGBM (cấu hình tốt nhất trong `baseline_search`, seed 42) và GAT (checkpoint seed 42): tìm node Normal bị đoán là tấn công, rồi trả lời (1) IP nào và IP đó có là node tấn công ở cửa sổ khác không, (2) độ thuần nhãn và nhãn lấy từ flow ra hay nhận, (3) tâm lớp gần nhất, (4) cách gán nhãn khác và ảnh hưởng (phản thực tế, KHÔNG đổi nhãn, KHÔNG huấn luyện lại).

**CPU, không dùng GPU.** Chỉ dùng train và val, không đọc test. Cần CSV gốc để dựng lại nhãn flow nên gắn dataset `athena21/netflow-v3-datasets`; gắn output của notebook 04, 11 (đồ thị), 07, 14 (checkpoint GAT), 05, 13 (log tìm siêu tham số). Bật Internet.

Đầu ra: `fpr_analysis_<bộ>.json`, `gd5_fpr_<bộ>_{ip,purity,centroid,relabel}.png`, `run_info.json`.

In [ ]:
import glob, json, os, resource, subprocess, sys, time
from contextlib import contextmanager

WORK = "/kaggle/working"
REPO = "/tmp/gnn-iot-ids"          # ngoài /kaggle/working để không lẫn vào output
INFO = {"steps_s": {}, "device": "CPU", "gpu": None}
T_START = time.time()


def peak_ram_gb():
    """RAM đỉnh (GB) của tiến trình notebook và mọi tiến trình con đã kết thúc (ru_maxrss tính bằng KB trên Linux)."""
    kb = max(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss, resource.getrusage(resource.RUSAGE_CHILDREN).ru_maxrss)
    return round(kb / 2**20, 2)


def save_info():
    INFO["total_s"] = round(time.time() - T_START, 1)
    INFO["peak_ram_gb"] = peak_ram_gb()
    with open(f"{WORK}/run_info.json", "w", encoding="utf-8") as f:
        json.dump(INFO, f, ensure_ascii=False, indent=1)


@contextmanager
def step(name):
    t0 = time.time()
    try:
        yield
    finally:
        INFO["steps_s"][name] = round(time.time() - t0, 1)
        save_info()                # ghi sau mỗi bước, lỗi giữa chừng vẫn còn run_info.json
        print(f"--- {name}: {INFO['steps_s'][name]} s, RAM đỉnh tới nay {INFO['peak_ram_gb']} GB")


def run(cmd, ok=(0,), **kw):
    r = subprocess.run(cmd, capture_output=True, text=True, **kw)
    print(r.stdout[-9000:], end="")
    print(r.stderr[-2500:], end="", file=sys.stderr)
    print(f"[mã thoát {r.returncode}]")
    if r.returncode not in ok:
        raise RuntimeError(f"Lệnh lỗi: {' '.join(cmd)}")
    return r


with step("clone_va_cai_goi"):
    run(["rm", "-rf", REPO])
    run(["git", "clone", "--depth", "1", "https://github.com/minucedev/gnn-iot-ids.git", REPO])
    run(["git", "-C", REPO, "log", "-1", "--format=%H %s"])
    INFO["commit"] = subprocess.run(["git", "-C", REPO, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
    with open(f"{REPO}/requirements.txt") as f:
        reqs = [l.strip() for l in f if l.strip() and not l.startswith("#")]
    keep = [r for r in reqs if r.split("==")[0].lower() not in {"torch", "jupyter", "ipykernel"}]
    run([sys.executable, "-m", "pip", "install", "-q", *keep])

import torch
if torch.cuda.is_available():
    INFO["device"], INFO["gpu"] = "GPU", torch.cuda.get_device_name(0)
INFO["cpu_count"] = os.cpu_count()
print("Commit:", INFO["commit"], "| máy:", INFO["device"], INFO["gpu"], "| CPU:", INFO["cpu_count"])
assert INFO["device"] == "CPU", "Notebook này chạy CPU; không bật GPU"

## 1. pytest cho phần phân tích (dừng nếu lỗi)

In [ ]:
with step("pytest"):
    r = run([sys.executable, "-m", "pytest", "-q", "tests/test_fpr.py"], cwd=REPO)
    INFO["pytest"] = r.stdout.strip().splitlines()[-1]

## 2. Phân tích từng bộ

In [ ]:
JOBS = [("nf_ton_iot_v3", "NF-ToN-IoT-v3.csv"), ("nf_bot_iot_v3", "NF-BoT-IoT-v3.csv")]
for ds, fn in JOBS:
    csv = next(glob.iglob(f"/kaggle/input/**/{fn}", recursive=True))
    data_dir = os.path.dirname(next(glob.iglob(f"/kaggle/input/**/{ds}/split.json", recursive=True)))
    search = next(glob.iglob(f"/kaggle/input/**/baseline_search_{ds}.json", recursive=True))
    ckpt = next(glob.iglob(f"/kaggle/input/**/{ds}_gat_s42.pt", recursive=True))
    print("=====", ds, "\n  csv:", csv, "\n  đồ thị:", data_dir, "\n  search:", search, "\n  ckpt:", ckpt)
    with step(f"fpr_{ds}"):
        run([sys.executable, "-I", "scripts/fpr_analysis.py", data_dir, ds, csv, search, ckpt, WORK], cwd=REPO)

## 3. Tóm tắt và run_info.json

In [ ]:
for ds, _ in JOBS:
    j = json.load(open(f"{WORK}/fpr_analysis_{ds}.json", encoding="utf-8"))
    INFO[f"check_label_match_{ds}"] = j["check_recomputed_label_matches_graph"]
    INFO[f"fpr_{ds}"] = {m: round(v["rates"]["fpr"], 4) for m, v in j["models"].items()}
    print(ds, "khớp nhãn dựng lại:", j["check_recomputed_label_matches_graph"], "| FPR:", INFO[f"fpr_{ds}"])
INFO["files"] = sorted(f for f in os.listdir(WORK) if f.endswith((".json", ".png")))
save_info()
print(open(f"{WORK}/run_info.json", encoding="utf-8").read())